# Successful Attempt at Downloading DHW Data from NOAA's CRW

I encountered difficulties downloading the Degree Heating Week (DHW) data from NOAA's Coral Reef Watch dataset.

To work around this, I ran four different download methods simultaneously. This method succeeded for the majority of the timeline but excluded 1986.

# Technique Used

I initially tried using the same technique I had implemented for the SST data:

https://github.com/kduran-stack/climate_collection_kd/blob/draft-plots1/Untitled0.ipynb

 Unfortunately, splitting 40 years of data into four decade-long CSVs all at once did not work. What did work, however, was downloading each decade individually rather than running all the code together.

Additionally, since some years downloaded out of order, I added a progress tracker so I could monitor when each year finished downloading.

In [ ]:
#this is attempt four it worked well

import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request
from urllib.error import HTTPError, URLError
import pandas as pd

# Hollywood, FL Coral Reef Grid
LAT = 26.125
LON = -80.075

#making a folder in google colab "fast chunks"
os.makedirs("dhw_fast_chunks", exist_ok=True)

years = list(range(1987,1996)) #had to change year range since 1986 kept throwing an error

def fetch_year(year):
    start = f"{year}-01-01T12:00:00Z"
    end = f"{year}-12-31T12:00:00Z"
    chunk_file = f"dhw_fast_chunks/dhw_{year}.csv"

    # Reuse any years that already finished before you stopped the previous run
    if os.path.exists(chunk_file) and os.path.getsize(chunk_file) > 100:
        return year, chunk_file

    url = (
        f"https://pae-paha.pacioos.hawaii.edu/erddap/griddap/dhw_5km.csv?"
        f"CRW_DHW[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_HOTSPOT[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_BAA[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})]"
    )

    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

    for attempt in range(4):
        try:
            with urllib.request.urlopen(req, timeout=90) as resp, open(chunk_file, "wb") as out:
                out.write(resp.read())
            return year, chunk_file
        except (HTTPError, URLError, TimeoutError) as e:
            if attempt < 3:
                time.sleep(2 * (attempt + 1))
            else:
                raise RuntimeError(f"Failed year {year}: {e}")

results = {}
print("Testing Decade 1 (1986–1995) via PacIOOS mirror...")
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(fetch_year, y): y for y in years}
    for i, future in enumerate(as_completed(futures), 1):
        yr, path = future.result()
        results[yr] = path
        print(f"Completed {yr} ({i}/{len(years)})")

dfs = [
    pd.read_csv(results[y], skiprows=[1], parse_dates=["time"], index_col="time")
    for y in sorted(results)
]

df_decade1 = pd.concat(dfs).dropna(subset=["CRW_DHW"])
df_decade1.index = df_decade1.index.normalize()

output_csv = "hollywood_reefs_1986_1995_dhw.csv"
df_decade1.to_csv(output_csv)
print(f"\nDone! Total daily rows for 1986–1995: {len(df_decade1):,}")
df_decade1.head()

Testing Decade 1 (1986–1995) via PacIOOS mirror...


In [ ]:
#continuing attempt 4 for the next decade


LAT = 26.125
LON = -80.075

os.makedirs("dhw_fast_chunks", exist_ok=True)

# Decade 2 (1996–2005 = 10 years)
years = list(range(1996, 2006))

def fetch_year(year):
    start = f"{year}-01-01T12:00:00Z"
    end = f"{year}-12-31T12:00:00Z"
    chunk_file = f"dhw_fast_chunks/dhw_{year}.csv"

    # Reuse any years that already finished if you have to re-run
    if os.path.exists(chunk_file) and os.path.getsize(chunk_file) > 100:
        return year, chunk_file

    url = (
        f"https://pae-paha.pacioos.hawaii.edu/erddap/griddap/dhw_5km.csv?"
        f"CRW_DHW[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_HOTSPOT[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_BAA[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})]"
    )

    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for attempt in range(4):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp, open(chunk_file, "wb") as out:
                out.write(resp.read())
            return year, chunk_file
        except (HTTPError, URLError, TimeoutError) as e:
            if os.path.exists(chunk_file):
                os.remove(chunk_file)  # Remove partial file on timeout
            if attempt < 3:
                time.sleep(3 * (attempt + 1))
            else:
                raise RuntimeError(f"Failed year {year}: {e}")

results = {}
print("Downloading Decade 2 (1996–2005) via PacIOOS mirror...")
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(fetch_year, y): y for y in years}
    for i, future in enumerate(as_completed(futures), 1):
        yr, path = future.result()
        results[yr] = path
        print(f"Completed {yr} ({i}/{len(years)})")

dfs = [
    pd.read_csv(results[y], skiprows=[1], parse_dates=["time"], index_col="time")
    for y in sorted(results)
]

df_decade2 = pd.concat(dfs).dropna(subset=["CRW_DHW"])
df_decade2.index = df_decade2.index.normalize()

output_csv = "hollywood_reefs_1996_2005_dhw.csv"
df_decade2.to_csv(output_csv)
print(f"\nDone! Total daily rows for 1996–2005: {len(df_decade2):,}")
df_decade2.head()

Completed 1997 (1/10)
Completed 1996 (2/10)
Completed 1998 (3/10)
Completed 1999 (4/10)
Completed 2000 (5/10)
Completed 2001 (6/10)
Completed 2002 (7/10)
Completed 2003 (8/10)
Completed 2004 (9/10)
Completed 2005 (10/10)

Done! Total daily rows for 1996–2005: 3,652


,latitude,longitude,CRW_DHW,CRW_HOTSPOT,CRW_BAA
time,,,,,
1996-01-01 00:00:00+00:00,26.125,-80.075,0.0,-4.96,0
1996-01-02 00:00:00+00:00,26.125,-80.075,0.0,-5.15,0
1996-01-03 00:00:00+00:00,26.125,-80.075,0.0,-5.36,0
1996-01-04 00:00:00+00:00,26.125,-80.075,0.0,-5.70,0
1996-01-05 00:00:00+00:00,26.125,-80.075,0.0,-5.90,0


In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request
from urllib.error import HTTPError, URLError
import pandas as pd

# Hollywood Reefs offshore 5km ocean pixel
LAT = 26.125
LON = -80.075

os.makedirs("dhw_fast_chunks", exist_ok=True)

# Decade 3 (2006–2015 = 10 years)
years = list(range(2006, 2016))

def fetch_year(year):
    start = f"{year}-01-01T12:00:00Z"
    end = f"{year}-12-31T12:00:00Z"
    chunk_file = f"dhw_fast_chunks/dhw_{year}.csv"

    # Reuse any years that already finished if you have to re-run
    if os.path.exists(chunk_file) and os.path.getsize(chunk_file) > 100:
        return year, chunk_file

    url = (
        f"https://pae-paha.pacioos.hawaii.edu/erddap/griddap/dhw_5km.csv?"
        f"CRW_DHW[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_HOTSPOT[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_BAA[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})]"
    )

    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for attempt in range(4):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp, open(chunk_file, "wb") as out:
                out.write(resp.read())
            return year, chunk_file
        except (HTTPError, URLError, TimeoutError) as e:
            if os.path.exists(chunk_file):
                os.remove(chunk_file)  # Remove partial file on timeout
            if attempt < 3:
                time.sleep(3 * (attempt + 1))
            else:
                raise RuntimeError(f"Failed year {year}: {e}")

results = {}
print("Downloading Decade 3 (2006–2015) via PacIOOS mirror...")
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(fetch_year, y): y for y in years}
    for i, future in enumerate(as_completed(futures), 1):
        yr, path = future.result()
        results[yr] = path
        print(f"Completed {yr} ({i}/{len(years)})")

dfs = [
    pd.read_csv(results[y], skiprows=[1], parse_dates=["time"], index_col="time")
    for y in sorted(results)
]

df_decade3 = pd.concat(dfs).dropna(subset=["CRW_DHW"])
df_decade3.index = df_decade3.index.normalize()

# Save inside Colab only
output_csv = "hollywood_reefs_2006_2015_dhw.csv"
df_decade3.to_csv(output_csv)
print(f"\nDone! Total daily rows for 2006–2015: {len(df_decade3):,}")
df_decade3.head()

Completed 2006 (1/10)
Completed 2007 (2/10)
Completed 2008 (3/10)
Completed 2009 (4/10)
Completed 2011 (5/10)
Completed 2010 (6/10)
Completed 2013 (7/10)
Completed 2012 (8/10)
Completed 2014 (9/10)
Completed 2015 (10/10)

Done! Total daily rows for 2006–2015: 3,652


,latitude,longitude,CRW_DHW,CRW_HOTSPOT,CRW_BAA
time,,,,,
2006-01-01 00:00:00+00:00,26.125,-80.075,0.0,-6.12,0
2006-01-02 00:00:00+00:00,26.125,-80.075,0.0,-5.79,0
2006-01-03 00:00:00+00:00,26.125,-80.075,0.0,-5.64,0
2006-01-04 00:00:00+00:00,26.125,-80.075,0.0,-5.82,0
2006-01-05 00:00:00+00:00,26.125,-80.075,0.0,-6.22,0


In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request
from urllib.error import HTTPError, URLError
import pandas as pd

# Hollywood Reefs offshore 5km ocean pixel
LAT = 26.125
LON = -80.075

os.makedirs("dhw_fast_chunks", exist_ok=True)

# Final stretch (2016–2026 = 11 years)
years = list(range(2016, 2027))

def fetch_year(year):
    start = f"{year}-01-01T12:00:00Z"
    # Use 'last' for 2026 so it grabs through the latest available satellite date without out-of-bounds errors
    end = "last" if year == 2026 else f"{year}-12-31T12:00:00Z"
    chunk_file = f"dhw_fast_chunks/dhw_{year}.csv"

    # Reuse any years that already finished if you have to re-run
    if os.path.exists(chunk_file) and os.path.getsize(chunk_file) > 100:
        return year, chunk_file

    url = (
        f"https://pae-paha.pacioos.hawaii.edu/erddap/griddap/dhw_5km.csv?"
        f"CRW_DHW[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_HOTSPOT[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})],"
        f"CRW_BAA[({start}):1:({end})][({LAT}):1:({LAT})][({LON}):1:({LON})]"
    )

    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for attempt in range(4):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp, open(chunk_file, "wb") as out:
                out.write(resp.read())
            return year, chunk_file
        except (HTTPError, URLError, TimeoutError) as e:
            if os.path.exists(chunk_file):
                os.remove(chunk_file)  # Remove partial file on timeout
            if attempt < 3:
                time.sleep(3 * (attempt + 1))
            else:
                raise RuntimeError(f"Failed year {year}: {e}")

results = {}
print("Downloading 2016–2026 via PacIOOS mirror...")
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(fetch_year, y): y for y in years}
    for i, future in enumerate(as_completed(futures), 1):
        yr, path = future.result()
        results[yr] = path
        print(f"Completed {yr} ({i}/{len(years)})")

dfs = [
    pd.read_csv(results[y], skiprows=[1], parse_dates=["time"], index_col="time")
    for y in sorted(results)
]

df_decade4 = pd.concat(dfs).dropna(subset=["CRW_DHW"])
df_decade4.index = df_decade4.index.normalize()

# Save inside Colab only
output_csv = "hollywood_reefs_2016_2026_dhw.csv"
df_decade4.to_csv(output_csv)
print(f"\nDone! Total daily rows for 2016–2026: {len(df_decade4):,}")
print(f"Latest available date: {df_decade4.index.max().strftime('%Y-%m-%d')}")
df_decade4.tail()

Completed 2017 (1/11)
Completed 2016 (2/11)
Completed 2019 (3/11)
Completed 2020 (4/11)
Completed 2018 (5/11)
Completed 2021 (6/11)
Completed 2022 (7/11)
Completed 2024 (8/11)
Completed 2023 (9/11)
Completed 2026 (10/11)
Completed 2025 (11/11)

Done! Total daily rows for 2016–2026: 3,963
Latest available date: 2026-11-17


,latitude,longitude,CRW_DHW,CRW_HOTSPOT,CRW_BAA
time,,,,,
2026-11-13 00:00:00+00:00,26.125,-80.075,4.56,-1.56,0
2026-11-14 00:00:00+00:00,26.125,-80.075,4.40,-1.54,0
2026-11-15 00:00:00+00:00,26.125,-80.075,4.22,-1.78,0
2026-11-16 00:00:00+00:00,26.125,-80.075,4.04,-1.89,0
2026-11-17 00:00:00+00:00,26.125,-80.075,3.88,-2.11,0


In [ ]:
import glob
import os
import zipfile
import pandas as pd
from google.colab import files

#merge 40 year csvs
chunk_files = sorted(glob.glob("dhw_fast_chunks/dhw_*.csv"))
dfs = [
    pd.read_csv(f, skiprows=[1], parse_dates=["time"], index_col="time")
    for f in chunk_files
]

df_master = pd.concat(dfs).dropna(subset=["CRW_DHW"])
df_master.index = df_master.index.normalize()
df_master = df_master.sort_index()

master_csv = "hollywood_reefs_1987_2026_dhw_master.csv"
df_master.to_csv(master_csv)
print(f"Created master CSV with {len(df_master):,} daily rows ({df_master.index.min().strftime('%Y-%m-%d')} to {df_master.index.max().strftime('%Y-%m-%d')}).")

# Bundle the master CSV, decade CSVs, and all 40 annual chunk files into one ZIP
zip_filename = "hollywood_reefs_1987_2026_all_files.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add all top-level CSVs (master, decades, and regional file)
    for top_csv in glob.glob("*.csv"):
        zf.write(top_csv)
    # Add all individual year files inside dhw_fast_chunks/
    for chunk in chunk_files:
        zf.write(chunk)

print(f"Bundled {len(chunk_files)} annual files + summary CSVs into {zip_filename}.")

# download to computer
files.download(zip_filename)

# Backup Dataset Option

In the case that I could not get the ERDDAP data to download, I wanted a back up option.

I downloaded the NOAA Coral Reef Watch 5km Regional Virtual Station data for Southeast Florida.

Summary of Data (Written by Gemini):

Source: NOAA Coral Reef Watch's pre-computed Virtual Station time series.

Spatial Coverage: It is a regional composite covering the entire Southeast Florida reef tract, rather than a single GPS coordinate.

Time Span: 40+ years of daily data starting from 1985-01-01 (15,271 rows downloaded).

Variables Included: Sea Surface Temperature minimums and maximums (SST_MIN, SST_MAX), regional 90th percentile HotSpots (SST@90th_HS, SSTA@90th_HS), Degree Heating Weeks (DHW_from_90th_HS>1), and Bleaching Alert Area (BAA_7day_max).

In [ ]:
import pandas as pd

# Pre-computed daily 5km time series for Southeast Florida (covers Hollywood Reefs)
url = "https://coralreefwatch.noaa.gov/product/vs/data/southeast_florida.txt"

# Find the exact header row where the data table begins
import urllib.request
lines = urllib.request.urlopen(url).read().decode("utf-8").splitlines()
header_idx = next(i for i, line in enumerate(lines) if line.startswith("YYYY"))

# Load the entire 40-year daily time series in one instant pass
df_vs = pd.read_csv(url, skiprows=header_idx, sep=r"\s+")

# Build a clean YYYY-MM-DD datetime index
df_vs["time"] = pd.to_datetime(
    dict(year=df_vs["YYYY"], month=df_vs["MM"], day=df_vs["DD"])
)
df_vs = df_vs.set_index("time").drop(columns=["YYYY", "MM", "DD"])

# Save to a single CSV in Colab
df_vs.to_csv("southeast_florida_40yr_daily_crw.csv")
print(f"Downloaded {len(df_vs):,} daily rows in seconds!")
df_vs.head()

Downloaded 15,271 daily rows in seconds!


,SST_MIN,SST_MAX,SST@90th_HS,SSTA@90th_HS,90th_HS>0,DHW_from_90th_HS>1,BAA_7day_max
time,,,,,,,
1985-01-01,24.31,25.07,24.98,0.2665,0.0,0.0,0
1985-01-02,24.33,25.18,25.01,0.2565,0.0,0.0,0
1985-01-03,24.44,25.31,25.19,0.4023,0.0,0.0,0
1985-01-04,24.48,25.56,25.19,0.4729,0.0,0.0,0
1985-01-05,23.94,25.40,24.91,0.4803,0.0,0.0,0
